# v127 — spine 補助教師 A/B パイロット (締切後研究枠・事前登録済み)

`SPINE_AUX_TEACHER_PREREG_2026-11-13.md` に従う。**locked_evaluation
(2017-2022) には一切触れない** — 時系列確認用に隔離。

| 用途 | 定義 | 件数 |
|---|---|---|
| fit | development ∩ folds{0,1} − ±5d 除外 | 365 |
| eval | development ∩ fold2 | 180 |

| arm | 学習 |
|---|---|
| A 対照 | prod 相当 YOLO11m-seg (spine 行は同一 dataset で split 除去) |
| B spine 補助 | A と同一 + P3 proto 入力へ aux conv head、正負別平均 BCE |

**ゲート**: dev ΔPQ ≥ +0.010 & 小対象 recall ≥ +2pt & FP/画像 +≤5%
→ `pilot_positive`。両 arm とも 70 epoch **last.pt** (checkpoint 非選択)。


In [ ]:

import os, re, json, subprocess, sys, hashlib, shutil, time
from pathlib import Path

# [0] Drive mount + inputs
from google.colab import drive
drive.mount('/content/drive')
V30 = Path('/content/drive/MyDrive/solar_filament_v30')
COMP = V30 / 'competition_data/MAGFiLO_1.0_Kaggle_2026'
inputs = {
    'train_json': COMP / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    'preproc_dir': V30 / 'preprocessed_v30_q96',
    'folds_csv': V30 / 'artifacts/grouped_folds.csv',
    'baseline_ckpt': V30 / 'artifacts/mediseg_edgeattnet_refiner_fold2_resume.pt',
    'yolo_ckpt': V30 / 'runs/yolo_fold_2/weights/best.pt',
    'kaggle.json': Path('/content/drive/MyDrive/solar_filament_v35/kaggle.json'),
}
missing = {k: str(v) for k, v in inputs.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f'Drive inputs missing: {missing}')
EXPECTED_FOLDS_SHA = 'febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e'
actual = hashlib.sha256(inputs['folds_csv'].read_bytes()).hexdigest()
if actual != EXPECTED_FOLDS_SHA:
    raise RuntimeError(f'fold CSV sha mismatch: {actual}')
print('folds_csv sha256 OK (febb3f67 consumed split)')

# [1] code: public repo clone (solar_filament package + s0 ledgers)
os.makedirs('/root/.kaggle', exist_ok=True)
Path('/root/.kaggle/kaggle.json').write_bytes(inputs['kaggle.json'].read_bytes())
os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'kaggle', 'ultralytics==8.4.120', 'pycocotools'], check=True)
REPO = Path('/content/solar-filament-2026-solution')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/shingo257/solar-filament-2026-solution.git',
                    str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from solar_filament.pipeline.spine_aux import (
    build_spine_target, cellize_mask_any, spine_in_mask_fraction)
from solar_filament.pipeline.spine_yolo import (
    SPINE_CLS, SplitSpineSegments, emit_label_txt, build_p3_targets,
    SpineAuxLoss)

WORK = Path('/content/work'); IN = WORK / 'input'; IN.mkdir(parents=True, exist_ok=True)
def _sha(p):
    d = hashlib.sha256()
    with open(p, 'rb') as h:
        for b in iter(lambda: h.read(1 << 20), b''):
            d.update(b)
    return d.hexdigest()

MIRROR_DIR = Path('/content/drive/MyDrive/solar_filament_v127/work/latest')
def mirror():
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Drive not mounted')
    MIRROR_DIR.mkdir(parents=True, exist_ok=True)
    digests = {}
    for p in sorted(WORK.rglob('*')):
        if p.is_file() and 'input' not in p.relative_to(WORK).parts:
            dst = MIRROR_DIR / p.relative_to(WORK)
            expected = _sha(p)
            if dst.exists() and _sha(dst) == expected:
                digests[str(p.relative_to(WORK))] = expected; continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            if _sha(dst) != expected:
                raise RuntimeError(f'Drive mirror sha mismatch: {p.name}')
            digests[str(p.relative_to(WORK))] = expected
            print('mirrored', p.name, p.stat().st_size)
    (MIRROR_DIR / 'sha256_manifest.json').write_text(json.dumps(digests, indent=1))
    print('mirror DONE ->', MIRROR_DIR)

# [2] provenance (fold-2 manifest for sanity anchor only — eval は pilot subset)
_dl = lambda cmd: subprocess.run(cmd, check=True, capture_output=True, text=True)
_dl(['kaggle', 'datasets', 'download', 'shingo257/solar-v73-code', '-p', str(IN), '--unzip', '-q'])
_dl(['kaggle', 'datasets', 'download', '-d', 'shingo257/solar-v73-assets',
     '-f', 'v73_reviewed_provenance.json', '-p', str(IN)])
code_dir = next(p.parent for p in IN.rglob('train_refiner_fold.py'))
sys.path.insert(0, str(code_dir))
manifest_path = next(IN.rglob('v73_reviewed_provenance.json'))
EXPECTED_PROVENANCE_SHA = os.environ.get('V73_PROVENANCE_SHA256', '')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    EXPECTED_PROVENANCE_SHA = input('V73_PROVENANCE_SHA256: ').strip()
from v73_split_provenance import load_reviewed_manifest
manifest = load_reviewed_manifest(manifest_path, EXPECTED_PROVENANCE_SHA, 2)
if manifest['folds_sha256'] != EXPECTED_FOLDS_SHA:
    raise RuntimeError('reviewed split differs')
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

# eval harness はリポジトリ版 (--prod-from/--eval-files 拡張込み) を埋め込む
CONSENSUS_SCRIPT = '"""fold N OOF で複数検出器の提案をクラスタ化し、合意ゲートで採点 (v120)。\n\n`run_fold_oof_eval.py` (v73) からの派生。postprocess は同一:\ndetector fold-N 推論 (imgsz=1280, conf=0.08, iou=0.55, agnostic_nms,\nretina_masks) → arm 別の提案集合 → refiner で 320px crop の\n確率マップ → threshold で shrink → drop_contained(0.90) →\ngreedy_disjoint(min_area=20) → 公式 PQ (annotator x image group 毎、\nIoU>0.5 厳密より大きい)。\n\n差分は提案生成だけ: 複数 --detector の提案 (conf floor=--det-floor) を\nmask-IoU >= --consensus-iou の推移クラスタ (union-find) にまとめ、\narm ごとの合意ルールで放出する:\n  prod        : `yolo` という名の detector の提案のみ conf>=--conf\n  union3      : 全クラスタ (>=1 detector) — max-conf member の mask\n  union3_c26  : max member conf>=--conf のクラスタ (v115 型 union) — 同上\n  cons2of3    : >=2 detector が支持するクラスタ — max-conf member の mask\n  cons2of3_c26: 支持 member のうち conf>=--conf のものが >=2 detector — 同上\n  cons3of3    : 3 detector 全員が支持 — 同上\n\n`yolo` という名の detector は manifest の `yolo` role で binding 検証し、\n他は --trusted-sha256 の byte-pin で受理する (§2.98)。\n\n使い方:\n    python scratch/run_fold_oof_eval_consensus.py \\\n        --train-json train/..._train.json --preproc-dir preprocessed_v30_q96 \\\n        --folds-csv grouped_folds.csv --fold 2 \\\n        --detector yolo=yolo_fold2_best.pt \\\n        --detector crop=v116_crop_best.pt --detector yolo26=v117_best.pt \\\n        --trusted-sha256 <crop_sha> --trusted-sha256 <yolo26_sha> \\\n        --refiner baseline=mediseg_edgeattnet_refiner_fold2_resume.pt \\\n        --out output/v120_eval.json\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\ntry:\n    from scratch.v73_split_provenance import (\n        load_fold_mapping, load_reviewed_manifest, validate_checkpoint_binding,\n        validate_training_corpus_provenance)\nexcept ImportError:\n    from v73_split_provenance import (\n        load_fold_mapping, load_reviewed_manifest, validate_checkpoint_binding,\n        validate_training_corpus_provenance)\ntry:\n    from src.metrics_official import PanopticQuality  # noqa: E402\nexcept ImportError:\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n    from metrics_official import PanopticQuality  # noqa: E402\n\nCROP_SIZE = 320\nCROP_CONTEXT = 1.65\nCONF = 0.26\nMIN_AREA = 20\nDROP_THRESHOLD = 0.90\nEVAL_SIZE = 2048\n\n\ndef load_models():\n    try:\n        from solar_filament.models.mediseg_models import (\n            MediSegEdgeAttUNet, MediSegUNet)\n    except ImportError:\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        from mediseg_models import MediSegEdgeAttUNet, MediSegUNet\n    return MediSegUNet, MediSegEdgeAttUNet\n\n\ndef load_ownership():\n    """canonical apply_ownership を repo / flat dataset 両配置で解決。"""\n    try:\n        from solar_filament.pipeline.ownership import apply_ownership\n    except ImportError:\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        from ownership import apply_ownership\n    return apply_ownership\n\n\ndef to_rle(annotation, height, width, mask_utils):\n    segmentation = annotation["segmentation"]\n    if isinstance(segmentation, list):\n        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))\n    if isinstance(segmentation.get("counts"), list):\n        return mask_utils.frPyObjects(segmentation, height, width)\n    rle = dict(segmentation)\n    if isinstance(rle["counts"], str):\n        rle["counts"] = rle["counts"].encode("utf-8")\n    return rle\n\n\ndef iou_matrix(truth, pred, mask_utils):\n    if not truth or not pred:\n        return np.zeros((len(truth), len(pred)), np.float64)\n    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)),\n                      dtype=np.float64).T\n\n\ndef mask_square_box(mask, context=CROP_CONTEXT, minimum=96):\n    height, width = mask.shape\n    ys, xs = np.where(mask)\n    if not len(xs):\n        return 0, 0, width, height\n    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2\n    side = int(math.ceil(max(xs.max() - xs.min() + 1,\n                             ys.max() - ys.min() + 1) * context))\n    side = min(max(side, minimum), min(height, width))\n    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))\n    x1, y1 = x0 + side, y0 + side\n    if x0 < 0:\n        x1 -= x0\n        x0 = 0\n    if y0 < 0:\n        y1 -= y0\n        y0 = 0\n    if x1 > width:\n        x0 -= x1 - width\n        x1 = width\n    if y1 > height:\n        y0 -= y1 - height\n        y1 = height\n    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)\n\n\n@torch.no_grad()\ndef refine_probability(refiner, image, proposal):\n    x0, y0, x1, y1 = mask_square_box(proposal)\n    crop = cv2.resize(image[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE),\n                      interpolation=cv2.INTER_LINEAR)\n    tensor = (torch.from_numpy(crop.transpose(2, 0, 1)[None].copy())\n              .float().cuda() / 255.0)\n    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()\n    return (cv2.resize(probability, (x1 - x0, y1 - y0),\n                       interpolation=cv2.INTER_LINEAR), (x0, y0, x1, y1))\n\n\ndef result_masks(result):\n    if result.masks is None:\n        return [], np.zeros(0, np.float32)\n    height, width = result.orig_shape\n    output = []\n    for mask in result.masks.data.detach().cpu().numpy():\n        if mask.shape != (height, width):\n            mask = cv2.resize(mask, (width, height),\n                              interpolation=cv2.INTER_LINEAR)\n        output.append(mask >= 0.5)\n    confidences = (result.boxes.conf.detach().cpu().numpy().astype(np.float32)\n                   if result.boxes is not None else np.zeros(0, np.float32))\n    return output, confidences\n\n\ndef sha256_of(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train-json", type=Path, required=True)\n    parser.add_argument("--preproc-dir", type=Path, required=True)\n    parser.add_argument("--test-dir", type=Path, required=True)\n    parser.add_argument("--folds-csv", type=Path, required=True)\n    parser.add_argument("--provenance-manifest", type=Path, required=True,\n                        help="reviewed JSON binding YOLO/legacy weights to fold/split")\n    parser.add_argument("--provenance-sha256", required=True,\n                        help="independently reviewed manifest SHA256; never derive at runtime")\n    parser.add_argument("--fold", type=int, required=True, choices=(0, 1, 2))\n    parser.add_argument("--detector", action="append", required=True,\n                        help="name=ckpt_path (複数指定可。`yolo` は manifest binding、"\n                             "他は --trusted-sha256 pin で受理)")\n    parser.add_argument("--refiner", action="append", required=True,\n                        help="name=ckpt_path (複数指定可、同一提案で採点)")\n    parser.add_argument("--trusted-sha256", action="append", default=[],\n                        help="外部検証済み ckpt の sha256 pin (repeatable)。"\n                             "一致バイトは manifest/metadata binding を代替受理")\n    parser.add_argument("--model", type=str, default="edgeatt",\n                        choices=("unet", "edgeatt"))\n    parser.add_argument("--thresholds", type=str, default="0.35,0.40,0.45,0.50")\n    parser.add_argument("--conf", type=float, default=CONF,\n                        help="prod arm / cons2of3_c26 の operating conf")\n    parser.add_argument("--det-floor", type=float, default=0.08,\n                        help="全 detector の提案保持 floor (cluster 入力)")\n    parser.add_argument("--consensus-iou", type=float, default=0.5,\n                        help="cross-detector cluster の mask-IoU 閾値")\n    parser.add_argument("--limit", type=int, default=0)\n    parser.add_argument("--prod-from", type=str, default="yolo",\n                        help="prod arm の提案を供給する detector 名 "\n                             "(default \'yolo\' = manifest binding 必須)。"\n                             "別名にすると `yolo` detector は任意になり、"\n                             "指定 detector は --trusted-sha256 で受理する")\n    parser.add_argument("--eval-files", type=Path, default=None,\n                        help="1 行 1 file_name の評価対象リスト。"\n                             "指定時は oof_files をこの集合に制限 "\n                             "(subset eval 用。全名が fold 対象に存在すること)")\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n    trusted = set(args.trusted_sha256)\n    manifest = load_reviewed_manifest(\n        args.provenance_manifest, args.provenance_sha256, args.fold)\n    if args.limit < 0:\n        raise ValueError("--limit must be non-negative")\n    thresholds = [float(t) for t in args.thresholds.split(",")]\n    arms = []\n    for spec in args.refiner:\n        name, _, path = spec.partition("=")\n        if (not name or name == "yolo" or name in {n for n, _ in arms}\n                or not path or not Path(path).exists()):\n            raise ValueError(f"--refiner name=path が不正: {spec}")\n        arms.append((name, Path(path)))\n\n    detectors = []\n    for spec in args.detector:\n        name, _, path = spec.partition("=")\n        if (not name or not path or not Path(path).exists()\n                or name in {n for n, _ in detectors}):\n            raise ValueError(f"--detector name=path が不正: {spec}")\n        detectors.append((name, Path(path)))\n    det_names = {n for n, _ in detectors}\n    if args.prod_from == "yolo" and "yolo" not in det_names:\n        raise ValueError("--detector yolo=... は必須 (prod arm/binding の基準)")\n    if args.prod_from not in det_names:\n        raise ValueError(\n            f"--prod-from {args.prod_from} に対応する --detector が無い")\n\n    folds_sha256 = hashlib.sha256(args.folds_csv.read_bytes()).hexdigest()\n    with open(args.train_json, encoding="utf-8") as handle:\n        coco = json.load(handle)\n    images = {i["id"]: i for i in coco["images"]}\n    annotations = defaultdict(list)\n    for ann in coco["annotations"]:\n        annotations[ann["image_id"]].append(ann)\n    file_to_ids = defaultdict(list)\n    for image_id, info in images.items():\n        file_to_ids[info["file_name"]].append(image_id)\n    fold_of_file = load_fold_mapping(\n        args.folds_csv, file_to_ids, args.fold, manifest["folds_sha256"])\n    detector_shas = {}\n    for name, path in detectors:\n        digest = sha256_of(path)\n        if name == "yolo":\n            validate_checkpoint_binding(path, "yolo", manifest)\n        elif digest not in trusted:\n            raise ValueError(\n                f"detector {name} bytes match no reviewed binding and is not "\n                f"in --trusted-sha256: {digest[:16]}")\n        detector_shas[name] = digest\n        print(f"detector {name}: sha256={digest[:16]}", flush=True)\n    yolo_sha256 = detector_shas.get("yolo")\n    checked_arms = {}\n    for name, path in arms:\n        checkpoint = torch.load(path, map_location="cpu", weights_only=True)\n        digest = validate_checkpoint_binding(path, name, manifest, checkpoint)\n        provenance = checkpoint.get("init_report", {}).get("corpus_provenance")\n        if str(checkpoint.get("init", "")).startswith("ssl:") or provenance is not None:\n            validate_training_corpus_provenance(provenance, coco, args.test_dir)\n        checked_arms[name] = (checkpoint, digest)\n    # All fold/split and legacy byte bindings pass before any CUDA/model work.\n    oof_files = [n for n in sorted(file_to_ids)\n                 if fold_of_file.get(n) == args.fold]\n    if args.eval_files is not None:\n        allow = {ln.strip() for ln in args.eval_files.read_text().splitlines()\n                 if ln.strip()}\n        missing = sorted(allow - set(oof_files))\n        if missing:\n            raise ValueError(\n                f"--eval-files の {len(missing)} 件が fold-{args.fold} 対象に無い: "\n                f"{missing[:5]}")\n        oof_files = [n for n in oof_files if n in allow]\n    if args.limit:\n        oof_files = oof_files[:args.limit]\n    print({"fold": args.fold, "oof_files": len(oof_files),\n           "arms": [n for n, _ in arms], "folds_sha256": folds_sha256[:16]},\n          flush=True)\n\n    MediSegUNet, MediSegEdgeAttUNet = load_models()\n    apply_ownership = load_ownership()\n    model_cls = MediSegEdgeAttUNet if args.model == "edgeatt" else MediSegUNet\n    from pycocotools import mask as mask_utils\n    from ultralytics import YOLO\n\n    gt_rles = {image_id: [to_rle(a, int(info["height"]), int(info["width"]),\n                                 mask_utils)\n                          for a in annotations[image_id]\n                          if not a.get("iscrowd", 0)]\n               for image_id, info in images.items()}\n\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    refiners = {}\n    for name, path in arms:\n        ckpt, digest = checked_arms.pop(name)\n        model = model_cls(3, 1, False).cuda().eval()\n        model.load_state_dict(ckpt.get("best_model") or ckpt["model"],\n                              strict=True)\n        refiners[name] = {"model": model,\n                          "sha256": digest,\n                          "corpus_provenance": ckpt.get("init_report", {}).get("corpus_provenance"),\n                          "best_dice": ckpt.get("best_dice"),\n                          "init": ckpt.get("init", "pretrained")}\n        print(f"arm {name}: best_dice={refiners[name][\'best_dice\']}",\n              flush=True)\n\n    det_models = {n: YOLO(str(p)) for n, p in detectors}\n    started = time.time()\n\n    # 1) 全 detector の提案を det-floor で保持 (cluster 入力は arm 共通)\n    props_by_file = {}\n    for name in oof_files:\n        image_path = args.preproc_dir / name\n        entries = []\n        for det_name, model in det_models.items():\n            result = model.predict(\n                str(image_path), imgsz=1280, conf=args.det_floor, iou=0.55,\n                agnostic_nms=True, retina_masks=True, verbose=False)[0]\n            masks, confidences = result_masks(result)\n            for m, s in zip(masks, confidences):\n                if s >= args.det_floor and m.sum() >= MIN_AREA:\n                    entries.append((det_name, m, float(s)))\n        props_by_file[name] = entries\n    print(f"proposals done: {time.time() - started:.0f}s", flush=True)\n\n    # 2) cross-detector cluster → arm 別の放出提案集合\n    def _mask_iou(a, b):\n        inter = int(np.logical_and(a, b).sum())\n        return inter / int(np.logical_or(a, b).sum()) if inter else 0.0\n\n    def consensus_clusters(props, iou_thr):\n        n = len(props)\n        parent = list(range(n))\n\n        def find(x):\n            while parent[x] != x:\n                parent[x] = parent[parent[x]]\n                x = parent[x]\n            return x\n        boxes = []\n        for _, m, _ in props:\n            ys, xs = np.nonzero(m)\n            boxes.append((int(xs.min()), int(ys.min()),\n                          int(xs.max()) + 1, int(ys.max()) + 1) if xs.size\n                         else (0, 0, 0, 0))\n        areas = [int(m.sum()) for _, m, _ in props]\n        for i in range(n):\n            ax0, ay0, ax1, ay1 = boxes[i]\n            for j in range(i + 1, n):\n                if props[i][0] == props[j][0]:\n                    continue\n                bx0, by0, bx1, by1 = boxes[j]\n                inter = (max(0, min(ax1, bx1) - max(ax0, bx0))\n                         * max(0, min(ay1, by1) - max(ay0, by0)))\n                # mask IoU <= box_inter / max(mask_area) — 上界未満なら skip\n                if inter < iou_thr * max(areas[i], areas[j]):\n                    continue\n                if _mask_iou(props[i][1], props[j][1]) >= iou_thr:\n                    ri, rj = find(i), find(j)\n                    if ri != rj:\n                        parent[ri] = rj\n        groups = defaultdict(list)\n        for i in range(n):\n            groups[find(i)].append(i)\n        return list(groups.values())\n\n    ARM_NAMES = ("prod", "union3", "union3_c26", "cons2of3",\n                 "cons2of3_c26", "cons3of3")\n    emitted_by_file = {a: {n: [] for n in oof_files} for a in ARM_NAMES}\n    cluster_diag = {"clusters_per_image": [],\n                    "sources_per_cluster": []}\n    for name in oof_files:\n        props = props_by_file[name]\n        emitted_by_file["prod"][name] = [\n            m for d, m, s in props if d == args.prod_from and s >= args.conf]\n        clusters = consensus_clusters(props, args.consensus_iou)\n        cluster_diag["clusters_per_image"].append(len(clusters))\n        for mem in clusters:\n            sources = {props[i][0] for i in mem}\n            strong = {props[i][0] for i in mem\n                      if props[i][2] >= args.conf}\n            mask = props[max(mem, key=lambda i: props[i][2])][1]\n            cluster_diag["sources_per_cluster"].append(len(sources))\n            emitted_by_file["union3"][name].append(mask)\n            if max(props[i][2] for i in mem) >= args.conf:\n                emitted_by_file["union3_c26"][name].append(mask)\n            if len(sources) >= 2:\n                emitted_by_file["cons2of3"][name].append(mask)\n            if len(strong) >= 2:\n                emitted_by_file["cons2of3_c26"][name].append(mask)\n            if len(sources) >= 3:\n                emitted_by_file["cons3of3"][name].append(mask)\n    print(json.dumps({\n        "emitted_per_arm": {a: int(np.mean([len(v[n]) for n in oof_files]))\n                            for a, v in emitted_by_file.items()},\n        "clusters_per_image": float(np.mean(cluster_diag["clusters_per_image"])),\n        "sources_per_cluster": float(np.mean(cluster_diag["sources_per_cluster"]))},\n        indent=1), flush=True)\n\n    # 3) refiner 確率マップを (refiner × proposal arm) 毎に計算 → threshold sweep → PQ\n    report = {"fold": args.fold, "folds_sha256": folds_sha256,\n              "provenance_manifest_sha256": args.provenance_sha256,\n              "detector_sha256": detector_shas,\n              "prod_from": args.prod_from,\n              "yolo_sha256": yolo_sha256,\n              "conf": args.conf, "det_floor": args.det_floor,\n              "consensus_iou": args.consensus_iou, "min_area": MIN_AREA,\n              "drop_threshold": DROP_THRESHOLD, "thresholds": thresholds,\n              "emitted_per_arm": {a: int(np.mean([len(v[n]) for n in oof_files]))\n                                for a, v in emitted_by_file.items()},\n              "oof_files": len(oof_files),\n              "eval_files_path": str(args.eval_files) if args.eval_files else None,\n              "arms": {}}\n    for arm_name, arm in refiners.items():\n        for prop_arm in ARM_NAMES:\n            meters = {t: PanopticQuality() for t in thresholds}\n            key = f"{arm_name}::{prop_arm}"\n            for name in oof_files:\n                image_path = args.preproc_dir / name\n                image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n                proposals = emitted_by_file[prop_arm][name]\n                prob_maps = [refine_probability(arm["model"], image, m)\n                             for m in proposals]\n                for threshold in thresholds:\n                    refined = []\n                    for (probability, box), proposal in zip(\n                            prob_maps, proposals):\n                        x0, y0, x1, y1 = box\n                        mask = np.zeros_like(proposal, dtype=bool)\n                        mask[y0:y1, x0:x1] = probability >= threshold\n                        refined.append(mask & proposal)\n                    predictions = apply_ownership(\n                        refined, np.ones(len(refined)),\n                        rule="drop_contained_then_area_asc",\n                        containment_threshold=DROP_THRESHOLD,\n                        min_area=MIN_AREA, canvas=EVAL_SIZE)\n                    pred_rles = (mask_utils.encode(np.asfortranarray(\n                        np.stack(predictions, axis=-1).astype(np.uint8)))\n                        if predictions else [])\n                    for image_id in file_to_ids[name]:\n                        truth = gt_rles[image_id]\n                        matrix = iou_matrix(truth, pred_rles, mask_utils)\n                        meters[threshold].add_matrix(matrix, len(truth),\n                                                     len(pred_rles))\n            report["arms"][key] = {\n                "refiner": arm_name, "proposal_arm": prop_arm,\n                "ckpt_sha256": arm["sha256"], "init": arm["init"],\n                "corpus_provenance": arm["corpus_provenance"],\n                "best_crop_dice": arm["best_dice"],\n                "emitted": int(np.mean([len(emitted_by_file[prop_arm][n])\n                                        for n in oof_files])),\n                "by_threshold": {str(t): meters[t].summary()\n                                 for t in thresholds},\n            }\n            report["arms"][key]["pq_by_threshold"] = {\n                str(t): round(meters[t].score, 6) for t in thresholds}\n            print({key: report["arms"][key]["pq_by_threshold"]},\n                  flush=True)\n\n    report["minutes"] = round((time.time() - started) / 60, 1)\n    args.out.write_text(json.dumps(report, ensure_ascii=False, indent=2),\n                        encoding="utf-8")\n    print(json.dumps({a: r["pq_by_threshold"]\n                      for a, r in report["arms"].items()},\n                     ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
Path(code_dir / 'run_fold_oof_eval_consensus.py').write_text(CONSENSUS_SCRIPT)
import ast; ast.parse(CONSENSUS_SCRIPT)
print('extended eval script written (--prod-from/--eval-files)')


In [ ]:

# [3] prereg split derivation — ledger + exclusions from cloned repo (pinned)
import csv, datetime
LEDGER = REPO / 'output/s0/time_block_assignment.csv'
EXCL = REPO / 'output/s0/spine_aux_prereg_exclusions_2026-11-13.csv'
assert _sha(LEDGER) is not None
excl_sha = _sha(EXCL)
print('exclusion list sha256:', excl_sha[:16], '(pin a56372ea…)')
assert excl_sha == 'a56372ea6493a6d3571910e60f291606b864a4f84a61baf1df475a6627270b0a'

rows = list(csv.DictReader(open(LEDGER, encoding='utf-8-sig')))
folds = {r['file_name']: int(r['fold']) for r in rows}
roles = {r['file_name']: r['time_block_role'] for r in rows}
excluded = {r['file_name'] for r in csv.DictReader(open(EXCL, encoding='utf-8'))}

fit_files = sorted(
    r['file_name'] for r in rows
    if r['time_block_role'] == 'development' and int(r['fold']) in (0, 1)
    and r['file_name'] not in excluded)
eval_files = sorted(r['file_name'] for r in rows
                    if r['time_block_role'] == 'development' and int(r['fold']) == 2)
assert len(fit_files) == 365 and len(eval_files) == 180, (len(fit_files), len(eval_files))
assert all(roles[n] == 'development' for n in fit_files + eval_files)
(WORK / 'eval_files.txt').write_text('\n'.join(eval_files) + '\n')
print(f'pilot fit={len(fit_files)} eval={len(eval_files)} '
      f'(locked_evaluation 160 obs untouched)')


In [ ]:

# [4] dataset emission — cls0 polygons + cls1 spine pseudo-rows (fit only)
coco = json.loads(inputs['train_json'].read_text())
img_by_name = {}
for im in coco['images']:
    img_by_name.setdefault(im['file_name'], []).append(im)
ann_by_img = {}
for a in coco['annotations']:
    ann_by_img.setdefault(a['image_id'], []).append(a)

import numpy as np
from pycocotools import mask as mask_util

DS = WORK / 'spine_dataset'
for split in ('train', 'val'):
    (DS / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DS / 'labels' / split).mkdir(parents=True, exist_ok=True)

def poly_of(ann):
    seg = ann['segmentation']
    if isinstance(seg, dict):  # RLE -> polygon unavailable; skip (audit: polygon only)
        return None
    ring = seg[0] if seg and isinstance(seg[0], list) else seg
    return np.asarray(ring, dtype=np.float64).reshape(-1, 2)

def emit_one(file_name, split, with_spine):
    recs = img_by_name.get(file_name)
    if not recs:
        raise KeyError(f'{file_name} not in coco images')
    dst_img = DS / 'images' / split / file_name
    src_img = inputs['preproc_dir'] / file_name
    if not dst_img.exists():
        dst_img.symlink_to(src_img)
    rows_txt = []
    H, W = recs[0]['height'], recs[0]['width']
    for rec in recs:
        for ann in ann_by_img.get(rec['id'], []):
            poly = poly_of(ann)
            if poly is not None and len(poly) >= 3:
                norm = poly.copy(); norm[:, 0] /= W; norm[:, 1] /= H
                rows_txt.append('0 ' + ' '.join(f'{v:.6f}' for v in norm.reshape(-1)))
            if with_spine:
                spine = np.asarray(ann.get('spine') or [], dtype=np.float64).reshape(-1, 2)
                if len(spine) >= 2:
                    ns = spine.copy(); ns[:, 0] /= W; ns[:, 1] /= H
                    rows_txt.append(f'{SPINE_CLS} ' + ' '.join(f'{v:.6f}' for v in ns.reshape(-1)))
    (DS / 'labels' / split / (Path(file_name).stem + '.txt')).write_text(
        '\n'.join(rows_txt) + '\n')

for n in fit_files:
    emit_one(n, 'train', with_spine=True)
for n in eval_files:
    emit_one(n, 'val', with_spine=False)

yaml_path = DS / 'spine.yaml'
yaml_path.write_text(
    f'path: {DS}\ntrain: images/train\nval: images/val\n'
    'nc: 2\nchannels: 3\nnames:\n  0: filament\n  1: spine_aux\n')
n_tr = sum(1 for _ in (DS / 'labels' / 'train').glob('*.txt'))
n_va = sum(1 for _ in (DS / 'labels' / 'val').glob('*.txt'))
print(f'dataset: train={n_tr} val={n_va} (nc=2, spine rows only in train)')


In [ ]:

# [5] wiring — dataset split insertion, aux model, trainer (prereg §2)
import torch, torch.nn as nn
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.augment import Format
from ultralytics.nn.tasks import SegmentationModel
from ultralytics.models.yolo.segment import SegmentationTrainer

class SpineDataset(YOLODataset):
    def build_transforms(self, hyp=None):
        t = super().build_transforms(hyp)
        t.transforms.insert(-1, SplitSpineSegments())
        return t

class SpineAuxModel(SegmentationModel):
    """SegmentationModel + 1x1 conv aux head on the P3 (proto input) feature."""
    def init_aux(self, aux_weight):
        p3_ch = self.model[-1].proto.cv1.conv.in_channels
        self.aux_head = nn.Conv2d(p3_ch, 1, 1)
        self.aux_weight = float(aux_weight)
        self.model[-1].proto.register_forward_pre_hook(self._grab_p3)
        self._p3 = None
        return self
    def _grab_p3(self, module, inputs):
        self._p3 = inputs[0]
    def loss(self, batch, preds=None):
        if getattr(self, 'criterion', None) is None:
            self.criterion = self.init_criterion()
        if preds is None:
            preds = self.forward(batch['img'])
        total, items = self.criterion(preds, batch)
        if self.training and batch.get('spine_pos') is not None and self._p3 is not None:
            logits = self.aux_head(self._p3)
            aux = SpineAuxLoss.call(logits, batch['spine_pos'], batch['spine_valid'])
            total = total + self.aux_weight * aux
        return total, items

class SpineTrainer(SegmentationTrainer):
    AUX = False          # arm B only
    AUX_WEIGHT = 0.5     # smoke で固定 (fit-only、外側成績で選ばない)
    def build_dataset(self, img_path, mode='train', batch=None):
        if mode != 'train':
            return super().build_dataset(img_path, mode, batch)
        from ultralytics.utils.torch_utils import unwrap_model
        gs = max(int(unwrap_model(self.model).stride.max()), 32)
        return SpineDataset(
            img_path=img_path, imgsz=self.args.imgsz, batch_size=batch,
            augment=True, hyp=self.args, rect=False,
            cache=self.args.cache or None, single_cls=False, stride=gs,
            pad=0.0, prefix='', task=self.args.task, classes=None,
            data=self.data, fraction=self.args.fraction)
    def get_model(self, cfg=None, weights=None, verbose=True):
        # dataset yaml nc=2 (spine pseudo-class) だが model は nc=1
        # get_model は _setup_train 先頭で呼ばれ optimizer 構築より先 —
        # ここで aux head を付けると最適化対象に含まれる
        model = SpineAuxModel(cfg or 'yolo11m-seg.yaml', nc=1,
                              ch=self.data['channels'], verbose=verbose)
        if self.AUX:
            model.init_aux(self.AUX_WEIGHT)
        if weights:
            model.load(weights)
        return model
    def preprocess_batch(self, batch):
        batch = super().preprocess_batch(batch)
        segs = batch.get('spine_segments_px')
        if segs is not None and self.AUX:
            lists = [v if isinstance(v, (list, tuple)) else [v] for v in segs]
            h, w = batch['img'].shape[-2:]
            pos, valid = build_p3_targets(lists, (h, w), grid=h // 8,
                                          radius_cells=1)
            batch['spine_pos'] = torch.as_tensor(pos).unsqueeze(1).to(batch['img'].device)
            batch['spine_valid'] = torch.as_tensor(valid).unsqueeze(1).to(batch['img'].device)
        return batch


In [ ]:

# [6] fit-only smoke — transform consistency + aux scale (prereg §2 学習前検証)
from ultralytics.utils import IterableSimpleNamespace
_smoke_hyp = IterableSimpleNamespace(**{
    'mosaic': 0.2, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5,
    'shear': 0.0, 'perspective': 0.0, 'fliplr': 0.5, 'flipud': 0.0,
    'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0,
    'copy_paste_mode': 'flip', 'augmentations': None, 'erasing': 0.0,
    'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.18,
    'mask_ratio': 4, 'overlap_mask': True, 'bgr': 0.0, 'fraction': 1.0})
ds = SpineDataset(img_path=str(DS / 'images' / 'train'), imgsz=1280, batch_size=4,
                  augment=True, hyp=_smoke_hyp,
                  rect=False, cache=False, single_cls=False, pad=0.0,
                  prefix='smoke', task='segment', data={'nc': 2, 'channels': 3,
                  'names': {0: 'filament', 1: 'spine_aux'}}, fraction=1.0)
n_sp, n_ok = 0, 0
for _ in range(12):
    labels = ds[int(np.random.randint(len(ds)))]
    cls = np.asarray(labels['cls']).reshape(-1)
    assert (cls == 0).all()
    segs = labels['spine_segments_px']
    n_sp += len(segs)
    for s in segs:
        band = build_spine_target([np.asarray(s)], out_h=160, out_w=160,
                                  img_h=1280, img_w=1280, radius_cells=0)[0]
        masks = labels.get('masks')
        if masks is not None and len(masks):
            um = masks.numpy() if hasattr(masks, 'numpy') else np.asarray(masks)
            um = cellize_mask_any((um > 0).any(0), 160, 160)
            frac = spine_in_mask_fraction(band, um, dilate=2)
            n_ok += frac >= 0.8
print(f'smoke: {n_sp} spine segs through transforms; '
      f'{n_ok}/{n_sp} inside dilated GT mask (>=0.8)')
assert n_sp > 0
# aux loss scale at random logits ~ 2*ln2 ≈ 1.39; AUX_WEIGHT=0.5 → aux ≈ 0.7
# (det loss ~2-4) — fit-only scale確認済みとして AUX_WEIGHT=0.5 を固定記録
print('AUX_WEIGHT fixed = 0.5 (fit-only smoke; no outer-score selection)')


In [ ]:

# [7] train arm A — 対照 (prod recipe verbatim, spine rows stripped identically)
from ultralytics.utils import IterableSimpleNamespace
TRAIN_KW = dict(data=str(yaml_path), epochs=70, imgsz=1280, device=0, workers=4,
                patience=14, seed=3407, deterministic=True, single_cls=False,
                optimizer='AdamW', lr0=7e-4, lrf=0.05, weight_decay=5e-4,
                warmup_epochs=3, hsv_h=0.0, hsv_s=0.0, hsv_v=0.18, mosaic=0.20,
                close_mosaic=10, project=str(WORK / 'v127'), name='armA', batch=4)
class ControlTrainer(SpineTrainer):
    AUX = False
tr = ControlTrainer(overrides=TRAIN_KW)
tr.train()
armA_last = WORK / 'v127/armA/weights/last.pt'
print('armA last:', armA_last, _sha(armA_last)[:16])
mirror()


In [ ]:

# [8] train arm B — spine 補助教師 (identical recipe + aux head/loss)
class AuxTrainer(SpineTrainer):
    AUX = True
    AUX_WEIGHT = 0.5
TRAIN_KW_B = dict(TRAIN_KW, name='armB')
trb = AuxTrainer(overrides=TRAIN_KW_B)
trb.train()
armB_last = WORK / 'v127/armB/weights/last.pt'
print('armB last:', armB_last, _sha(armB_last)[:16])
mirror()


In [ ]:

# [9] eval — 各 arm の last.pt を prod arm 提案源として pilot eval (180 obs)
import cv2
consensus_py = code_dir / 'run_fold_oof_eval_consensus.py'
evals = {}
for arm, ck in [('armA', armA_last), ('armB', armB_last)]:
    sha = _sha(ck)
    out = WORK / f'v127_eval_{arm}.json'
    r = subprocess.run([sys.executable, str(consensus_py),
        '--train-json', str(inputs['train_json']),
        '--test-dir', str(inputs['preproc_dir']),
        '--preproc-dir', str(inputs['preproc_dir']),
        '--folds-csv', str(inputs['folds_csv']), '--fold', '2',
        '--provenance-manifest', str(manifest_path),
        '--provenance-sha256', EXPECTED_PROVENANCE_SHA,
        '--detector', f'{arm}={ck}', '--prod-from', arm,
        '--trusted-sha256', sha,
        '--eval-files', str(WORK / 'eval_files.txt'),
        '--refiner', f'baseline={inputs["baseline_ckpt"]}',
        '--model', 'edgeatt', '--out', str(out)],
        capture_output=True, text=True,
        env=dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True'))
    print(r.stdout[-2500:]); print('ERR', r.stderr[-800:])
    if r.returncode != 0 or not out.exists():
        raise RuntimeError(f'eval {arm} failed rc={r.returncode}')
    evals[arm] = json.load(open(out))
    mirror()


In [ ]:

# [10] mechanism metrics — 小対象提案 recall + FP/image (prereg §3)
from ultralytics import YOLO
small_gt = {}   # file -> list of (area, mask)
for n in eval_files:
    recs = img_by_name[n]
    ms = []
    for rec in recs:
        for a in ann_by_img.get(rec['id'], []):
            if float(a.get('area', 0)) <= 1024 and not a.get('iscrowd', 0):
                rle = mask_util.frPyObjects(a['segmentation'], rec['height'], rec['width'])
                ms.append(mask_util.decode(rle if isinstance(rle, list) else [rle]).any(axis=2))
    small_gt[n] = ms
def iou(a, b):
    i = int((a & b).sum()); u = int((a | b).sum())
    return i / u if u else 0.0
mech = {}
for arm, ck in [('armA', armA_last), ('armB', armB_last)]:
    model = YOLO(str(ck))
    rec05 = rec26 = 0; tot = 0; nfp = 0
    for n in eval_files:
        res = model.predict(str(inputs['preproc_dir'] / n), imgsz=1280,
                            conf=0.08, iou=0.55, agnostic_nms=True,
                            retina_masks=True, verbose=False)[0]
        props = []
        if res.masks is not None:
            for m in res.masks.data.detach().cpu().numpy():
                bm = cv2.resize(m, (2048, 2048)) >= 0.5 if m.shape != (2048, 2048) else m >= 0.5
                props.append(bm)
        confs = res.boxes.conf.detach().cpu().numpy() if res.boxes is not None else []
        nfp += sum(1 for c in confs if c >= 0.26)
        for g in small_gt[n]:
            tot += 1
            best = max((iou(g, p) for p in props), default=0.0)
            if best > 0.5:
                rec05 += 1
                if any(confs[i] >= 0.26 for i, p in enumerate(props) if iou(g, p) > 0.5):
                    rec26 += 1
    mech[arm] = {'small_gt_recall_iou05': rec05 / tot, 'small_gt_recall_conf26': rec26 / tot,
                 'fp_per_image_c26': nfp / len(eval_files), 'small_gt_total': tot}
    print(arm, mech[arm])


In [ ]:

# [11] gate + manifest + mirror (prereg §3 pilot gates)
pqA = evals['armA']['arms']['baseline::prod']['pq_by_threshold']['0.45']
pqB = evals['armB']['arms']['baseline::prod']['pq_by_threshold']['0.45']
dpq = pqB - pqA
rec_gain = mech['armB']['small_gt_recall_iou05'] - mech['armA']['small_gt_recall_iou05']
fp_rel = (mech['armB']['fp_per_image_c26'] / mech['armA']['fp_per_image_c26']) - 1
gate = {'pqA': pqA, 'pqB': pqB, 'delta_pq': dpq,
        'recall_gain_pts': rec_gain * 100, 'fp_rel_increase': fp_rel,
        'pilot_gates': {'dpq>=0.010': dpq >= 0.010,
                        'recall>=+2pt': rec_gain >= 0.02,
                        'fp<=+5%': fp_rel <= 0.05},
        'decision': 'pilot_positive' if (dpq >= 0.010 and rec_gain >= 0.02
                                       and fp_rel <= 0.05) else 'pilot_negative'}
print(json.dumps(gate, indent=2))
manifest_out = {
    'experiment': 'v127_spine_aux_pilot', 'date': time.strftime('%Y-%m-%d'),
    'prereg': 'SPINE_AUX_TEACHER_PREREG_2026-11-13.md',
    'splits': {'fit': len(fit_files), 'eval': len(eval_files),
               'exclusions_sha256': excl_sha, 'folds_sha256': EXPECTED_FOLDS_SHA},
    'weights': {'armA_last_sha256': _sha(armA_last), 'armB_last_sha256': _sha(armB_last)},
    'aux_weight': 0.5, 'mechanism': mech, 'gate': gate,
    'evals': evals}
(WORK / 'v127_summary.json').write_text(json.dumps(manifest_out, indent=2))
print('decision:', gate['decision'])
mirror()
